In [3]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# =====================================
# LOAD OUTPUTS
# =====================================

forecast = pd.read_csv(
    r"C:\Users\L\Desktop\forecast_results.csv"
)

retention = pd.read_csv(
    r"C:\Users\L\Desktop\retention_results.csv"
)

# =====================================
# MERGE
# =====================================

decision = forecast.merge(
    retention,
    on="wilaya",
    how="left"
)

# =====================================
# NORMALIZE
# =====================================

scaler = MinMaxScaler()

decision["Revenue Score"] = scaler.fit_transform(
    decision[["forecast_revenue"]]
)

decision["Retention Score"] = scaler.fit_transform(
    decision[["average_retention"]]
)

decision["Risk Score"] = 1 - scaler.fit_transform(
    decision[["forecast_risk"]]
)

# =====================================
# FINAL SCORE
# =====================================

decision["Investment Score"] = (
      0.5 * decision["Revenue Score"]
    + 0.3 * decision["Retention Score"]
    + 0.2 * decision["Risk Score"]
)

# =====================================
# RECOMMENDATION
# =====================================

def recommendation(row):

    if row["Investment Score"] >= 0.80:
        return "High Priority Investment"

    elif row["Investment Score"] >= 0.60:
        return "Moderate Investment"

    elif row["Investment Score"] >= 0.40:
        return "Focus on customer retention before expanding."

    return "Maintain current investment."

decision["Recommendation"] = decision.apply(
    recommendation,
    axis=1
)

decision = decision.sort_values(
    "Investment Score",
    ascending=False
)

print(decision)

decision.to_csv(
    r"C:\Users\L\Desktop\final_decision_report.csv",
    index=False
)

  wilaya  forecast_revenue  forecast_risk    best_model  average_retention  \
0  Alger      5.823269e+06  960220.434488  Holt-Winters           0.153827   

   customers  Revenue Score  Retention Score  Risk Score  Investment Score  \
0        936            0.0              0.0         1.0               0.2   

                 Recommendation  
0  Maintain current investment.  
